# Walkthrough: Verification-Gated Epistemic Exploration (VGEE)

This notebook walks through the VGEE implementation, connecting each paper section to code with runnable sanity checks.

**Paper**: Verification-Gated Epistemic Exploration: Resolving the Winner-Take-All Paradox in RLVR

We use tiny dimensions so everything runs on CPU instantly.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys
sys.path.insert(0, '..')

# Toy dimensions for CPU execution
BATCH_SIZE = 2
SEQ_LEN = 16
VOCAB_SIZE = 100
D_MODEL = 64  # Paper uses Llama-3-8B scale; we use 64 for walkthrough
print('Setup complete')

## 1. Token-Level Entropy (§4.1, Eq. 1)

> "We quantify epistemic uncertainty using token-level entropy. For each generated token $y_t$:"
>
> $$H_t = - \sum_{v \in \mathcal{V}} P_\theta(v \mid x_{<t}, y_{<t}) \log P_\theta(v \mid x_{<t}, y_{<t})$$
>
> — §4.1

In [ ]:
from src.utils import compute_token_entropy

# Simulate logits from a language model
logits = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)  # (batch, seq_len, vocab)

entropy = compute_token_entropy(logits)  # (batch, seq_len)
assert entropy.shape == (BATCH_SIZE, SEQ_LEN), f"Expected ({BATCH_SIZE}, {SEQ_LEN}), got {entropy.shape}"
assert (entropy >= 0).all(), "Entropy must be non-negative"
print(f"✓ Token entropy: shape {entropy.shape}, range [{entropy.min():.3f}, {entropy.max():.3f}]")
print(f"  Max possible entropy for vocab={VOCAB_SIZE}: {math.log(VOCAB_SIZE):.3f} nats")

## 2. Trajectory Uncertainty (§4.1, Eq. 2)

> "The trajectory uncertainty $U_\tau$ is defined as the maximum entropy observed during the reasoning phase, identifying the point of greatest confusion:"
>
> $$U_\tau = \max_{t \in \text{reasoning}} H_t$$
>
> — §4.1

In [ ]:
from src.utils import compute_trajectory_uncertainty

# Use the entropy from the previous cell
U_tau = compute_trajectory_uncertainty(entropy)  # (batch,)
assert U_tau.shape == (BATCH_SIZE,), f"Expected ({BATCH_SIZE},), got {U_tau.shape}"
assert (U_tau >= entropy.max(dim=-1).values - 1e-5).all(), "U_tau should be max over tokens"
print(f"✓ Trajectory uncertainty: shape {U_tau.shape}, values {U_tau.tolist()}")

## 3. Verification Gate (§4.2)

> "The 'Verification Gate' is triggered based on an uncertainty threshold $\delta$. If $U_\tau > \delta$, the sample is flagged for Epistemic Exploration."
>
> — §4.2

In [ ]:
from src.utils import verification_gate

# [UNSPECIFIED] δ threshold — using 1.0 nats as default
delta = 1.0
is_high_uncertainty = U_tau > delta  # (batch,)
print(f"Uncertainty values: {U_tau.tolist()}")
print(f"Threshold δ = {delta}")
print(f"High uncertainty flags: {is_high_uncertainty.tolist()}")
print(f"✓ Verification gate: {is_high_uncertainty.sum().item()}/{BATCH_SIZE} trajectories flagged")

## 4. Conditional KL-Regularizer (§4.3)

> **Case A: Discovery** ($U_\tau > \delta \land V(\tau) = \text{True}$)
> "The model found a correct solution in a region of high uncertainty. We want to reinforce this strongly."
> $\beta_{\text{eff}} = \beta_{\text{base}}$
>
> **Case B: Failed Exploration** ($U_\tau > \delta \land V(\tau) = \text{False}$)
> "To prevent the 'winner-take-all' suppression of this region... we apply a strict KL constraint."
> $\beta_{\text{eff}} = \kappa \cdot \beta_{\text{base}}$, where $\kappa \gg 1$
>
> **Case C: Exploitation** ($U_\tau \le \delta$)
> Standard RL optimization.
>
> — §4.3

In [ ]:
from src.loss import conditional_kl_coefficient

# [UNSPECIFIED] β_base=0.04, κ=10.0
beta_base = 0.04
kappa = 10.0

# Simulate all 3 cases
U_tau_test = torch.tensor([2.0, 2.0, 0.5])  # high, high, low uncertainty
verified = torch.tensor([True, False, True])  # correct, incorrect, correct
delta = 1.0

beta_eff = conditional_kl_coefficient(U_tau_test, verified, delta, beta_base, kappa)

print("Case A (Discovery):  U=2.0, V=True  → β_eff =", beta_eff[0].item(), f"(= β_base = {beta_base})")
print("Case B (Failed Exp): U=2.0, V=False → β_eff =", beta_eff[1].item(), f"(= κ·β_base = {kappa*beta_base})")
print("Case C (Exploit):    U=0.5, V=True  → β_eff =", beta_eff[2].item(), f"(= β_base = {beta_base})")

assert abs(beta_eff[0].item() - beta_base) < 1e-6, "Case A should use β_base"
assert abs(beta_eff[1].item() - kappa * beta_base) < 1e-6, "Case B should use κ·β_base"
assert abs(beta_eff[2].item() - beta_base) < 1e-6, "Case C should use β_base"
print("✓ All 3 conditional KL cases verified")

## 5. PPO Objective with Conditional KL (§4.4, Eq. 5)

> $$L(\theta) = \mathbb{E}_t \left[ L^{CLIP}(\theta) - c_1 L^{VF}(\theta) + \text{Conditional\_KL}(\theta) \right]$$
>
> "where the KL term is determined dynamically per trajectory based on the logic in Section 4.3"
>
> — §4.4

In [ ]:
from src.loss import ppo_clip_loss, value_function_loss

# Simulate PPO inputs
batch = 4
old_log_probs = torch.randn(batch)  # log π_old(a|s)
new_log_probs = old_log_probs + torch.randn(batch) * 0.1  # log π_θ(a|s)
advantages = torch.randn(batch)  # Â_t from GAE
returns = torch.randn(batch)  # R_t
values = torch.randn(batch)  # V(s)

# [UNSPECIFIED] ε_clip = 0.2 (standard PPO default)
clip_loss = ppo_clip_loss(old_log_probs, new_log_probs, advantages, clip_epsilon=0.2)
vf_loss = value_function_loss(values, returns)

print(f"PPO clip loss: {clip_loss.item():.4f}")
print(f"Value function loss: {vf_loss.item():.4f}")
print(f"✓ PPO components compute without errors")

## 6. GAE Advantage Estimation (§4.4)

> "We will use Generalized Advantage Estimation (GAE) with a discount factor $\gamma = 0.99$ and GAE parameter $\lambda = 0.95$."
>
> — §4.4 (from Methods section)

In [ ]:
from src.loss import compute_gae

# Simulate a trajectory
T = 10  # trajectory length
rewards = torch.randn(T)
values = torch.randn(T + 1)  # includes bootstrap value
dones = torch.zeros(T)  # no episode boundaries

# §4.4 — γ=0.99, λ=0.95
advantages, returns = compute_gae(rewards, values, dones, gamma=0.99, gae_lambda=0.95)

assert advantages.shape == (T,), f"Expected ({T},), got {advantages.shape}"
assert returns.shape == (T,), f"Expected ({T},), got {returns.shape}"
print(f"✓ GAE: advantages shape {advantages.shape}, returns shape {returns.shape}")
print(f"  Advantage stats: mean={advantages.mean():.3f}, std={advantages.std():.3f}")

## 7. Full VGEE Training Step

Putting it all together: one complete VGEE training step combining entropy tracking → verification gate → conditional KL → PPO update.

In [ ]:
# Simulate a complete VGEE training step with toy data
from src.loss import conditional_kl_coefficient

num_trajectories = 8
delta = 1.0
beta_base = 0.04
kappa = 10.0

# Step 1: Sample trajectories and compute entropy
trajectory_uncertainties = torch.rand(num_trajectories) * 3  # U_τ ∈ [0, 3]
verification_results = torch.rand(num_trajectories) > 0.3  # ~70% correct

# Step 2: Classify into Cases A, B, C
high_U = trajectory_uncertainties > delta
case_A = high_U & verification_results  # Discovery
case_B = high_U & ~verification_results  # Failed exploration
case_C = ~high_U  # Exploitation

# Step 3: Compute conditional KL coefficients
beta_eff = conditional_kl_coefficient(
    trajectory_uncertainties, verification_results, delta, beta_base, kappa
)

print(f"Trajectories: {num_trajectories}")
print(f"Case A (Discovery):        {case_A.sum().item()} trajectories → β_eff = {beta_base}")
print(f"Case B (Failed Exploration): {case_B.sum().item()} trajectories → β_eff = {kappa * beta_base}")
print(f"Case C (Exploitation):      {case_C.sum().item()} trajectories → β_eff = {beta_base}")
print(f"\n✓ Full VGEE step: {num_trajectories} trajectories classified and KL coefficients assigned")

## Common Pitfalls

1. **κ value**: Paper says "κ >> 1" but never gives the value. Too low (e.g., 2) won't preserve exploration; too high (e.g., 100) freezes the policy. We use κ=10 as default. [§4.3]

2. **δ threshold tuning**: The uncertainty threshold δ determines what counts as "high uncertainty." Paper says "tuned on validation" but doesn't specify the value. This is the most sensitive hyperparameter — start with median entropy of the initial policy. [§4.2]

3. **Entropy computation**: Must use natural log (nats), not log2 (bits). PyTorch's default log is natural. [§4.1, Eq. 1]

4. **KL direction**: The KL divergence is KL(π_θ || π_ref), measuring how far the new policy drifts from the reference. Reversing this gives a completely different optimization landscape. [§4.3]

5. **Verification cost**: Running Python interpreter for every high-uncertainty trajectory is expensive. The paper notes this triggers only when U_τ > δ, which should be a fraction of trajectories. [§4.2]

6. **Case B preservation**: The strict KL in Case B doesn't mean "don't update" — it means "update very conservatively." The policy still learns from failed explorations; it just doesn't collapse away from that region. [§4.3]